# Try a molecule against the historical record

Nothing to install — this runs in the browser.

**Run the setup cell first.** After that every cell below is self-contained:
run them in any order, as many times as you like. The only one you need is the
second — edit the SMILES in it and run it.

### What it is actually comparing against

A molecule cannot be judged in the abstract — *"is this good?"* has no referent.
What this answers is *"is this a good optimization of **that** starting compound?"*,
and that question the published literature does settle.

The reference is **measured experimental activity** from ChEMBL: 32 real optimization
campaigns from published papers, each a starting compound plus the improved analogs
medicinal chemists actually reached, with their measured potencies.

Everything the tool reports is either a ChEMBL measurement or an RDKit substructure
match. **No model gives an opinion anywhere** — no ADMET predictor, no docking, no
LLM judge. When the published record cannot settle a question, the tool says so
instead of guessing.

## Setup — run once (takes about a minute)

In [ ]:
!pip install -q rdkit
!pip install -q git+https://github.com/SinanGokmen/leadopt-bench.git

import pandas as pd
from IPython.display import HTML, display
from rdkit import Chem
from rdkit.Chem import Draw
from leadopt_bench import metrics as M, report as R

SERIES = {s["series_id"]: s for s in M.available_series()}
print(f"ready — {len(SERIES)} series available\n")
pd.set_option("display.max_colwidth", 64)
display(pd.DataFrame(SERIES.values())[
    ["series_id", "target", "endpoint", "lead_pchembl", "n_targets", "max_gain"]])

## Option A — paste any SMILES

**One input.** Put a SMILES in `MY_MOLECULE` and run the cell. Nothing else is
required: the campaign to compare against is picked automatically — whichever of
the 32 starting compounds your molecule most resembles — and the cell prints which
one it chose.

If your molecule looks like none of them, the cell says so instead of forcing a
comparison that would mean nothing. To override the choice, set `SERIES_ID` to any
id from the setup table.

The default below shortens the HDAC1 lead's eleven-carbon linker to seven carbons.
`MY_CHANGE` and `MY_REASONING` are optional — but if you write the reasoning, the
tool also checks whether the structures you mention are really in the molecule.

In [ ]:
# ===================== EDIT THIS: just paste a SMILES =====================
MY_MOLECULE = "CC(C)(C)c1cc(NC(=O)Nc2ccc(Oc3ccnc(-c4cnn(CCCCCCC(=O)NO)c4)c3)cc2)no1"
# ==========================================================================

# optional — leave as-is, or describe your change and why you expect it to help
MY_CHANGE    = ""
MY_REASONING = ""
SERIES_ID    = None   # None = pick the closest campaign automatically

score, smiles, source, s = M.evaluate_identifier(
    MY_MOLECULE, SERIES_ID, modification=MY_CHANGE, rationale=MY_REASONING)

print(f"compared against: {s['series_id']}  ({s['target']})")
print(f"its starting compound: {s['lead_smiles']}")
print(f"measured p{s['endpoint']} of that compound: {s['lead_pchembl']}")
print(f"chemists reached {s['n_targets']} better ones, best gain "
      f"{s['max_gain']} log units")
print(f"\nyour structure ({source}): {smiles}\n")
display(HTML(R.verdict_html(score, s)))

mols = [m for m in (Chem.MolFromSmiles(s["lead_smiles"]),
                    Chem.MolFromSmiles(smiles)) if m is not None]
legends = [f"starting compound  (p={s['lead_pchembl']})",
           f"yours  ({score.category})"][:len(mols)]
display(Draw.MolsToGridImage(mols, legends=legends, molsPerRow=2,
                             subImgSize=(430, 300)))

## Option B — paste a ChEMBL id

**One input**, same as Option A, but for a compound that already exists: paste its
ChEMBL id instead of drawing a SMILES. Useful when you are browsing ChEMBL and want
to know how a particular compound would have scored.

Ids already in the bundled series resolve offline and instantly; anything else is
fetched from the ChEMBL API once and then cached.

`CHEMBL6145741` is the most potent compound in that HDAC1 series — a sanity check,
since the tool should recognise it and report its measured value.

In [ ]:
# =================== EDIT THIS: just paste a ChEMBL id ====================
MY_MOLECULE = "CHEMBL6145741"
# ==========================================================================

MY_CHANGE    = ""
MY_REASONING = ""
SERIES_ID    = None   # None = pick the closest campaign automatically

score, smiles, source, s = M.evaluate_identifier(
    MY_MOLECULE, SERIES_ID, modification=MY_CHANGE, rationale=MY_REASONING)

print(f"compared against: {s['series_id']}  ({s['target']})")
print(f"its starting compound: {s['lead_smiles']}")
print(f"measured p{s['endpoint']} of that compound: {s['lead_pchembl']}")
print(f"chemists reached {s['n_targets']} better ones, best gain "
      f"{s['max_gain']} log units")
print(f"\nyour structure ({source}): {smiles}\n")
display(HTML(R.verdict_html(score, s)))

mols = [m for m in (Chem.MolFromSmiles(s["lead_smiles"]),
                    Chem.MolFromSmiles(smiles)) if m is not None]
legends = [f"starting compound  (p={s['lead_pchembl']})",
           f"yours  ({score.category})"][:len(mols)]
display(Draw.MolsToGridImage(mols, legends=legends, molsPerRow=2,
                             subImgSize=(430, 300)))

## Every verdict the tool can give

The colour reflects **how much the published record settles** — not whether a
molecule is good.

In [ ]:
display(HTML(R.legend_html()))

### The three checks behind the verdict

| Check | How it is decided | Why it is not an opinion |
|---|---|---|
| Is it a valid molecule? | RDKit parsing and sanitization, single fragment, mass range | a parser either accepts the structure or it does not |
| Was the scaffold kept? | Morgan fingerprint Tanimoto against the starting compound | the threshold is **calibrated from the series itself**: the least similar compound real chemists made from that lead |
| Is the reasoning consistent? | a 40-entry SMARTS catalogue of rings and groups; every structure the text names is looked for in the lead **or** the proposal | substructure matching is a fact about the molecule |

The third check is the mechanizable part of what a domain expert does when they
glance at a suggestion and immediately see it is about the wrong compound.

One number needs care: **change-type accuracy**. A model that only ever adds
fluorine, chlorine or a methyl scores **0.61** on it, because halogen scanning
genuinely does raise potency. So it is only readable against that null floor, which
is why the floor is printed next to the number.

## Optional: what an LLM proposed on these same series

Gemma 3 12B was asked the same question on all 32 series — 160 suggestions.
The table below is the queue that reaches a human: valid molecules that kept the
scaffold and whose reasoning is at least about the right compound, but which the
chemists never made. Only a domain expert can settle these.

In [ ]:
import importlib.resources as res

sug = pd.read_csv(res.files("leadopt_bench")
                  / "data/reports/suggestions_final_gemma3_12b.csv")

print("all 160 suggestions, by verdict:")
print(sug["category"].value_counts().to_string(), "\n")

queue = sug[(sug["category"] == "novel_plausible")
            & (~sug["describes_other_molecule"])]
print(f"reaching a human: {len(queue)} of {len(sug)} "
      f"({len(sug) - len(queue)} filtered automatically)")
queue[["target", "modification", "rationale", "smiles"]].head(10)

### What the filters threw out, and why

In [ ]:
# self-contained: reloads the table so this cell runs on its own
import importlib.resources as res
import pandas as pd

sug = pd.read_csv(res.files("leadopt_bench")
                  / "data/reports/suggestions_final_gemma3_12b.csv")

print("--- discarded: not a valid molecule ---")
print(sug[sug["category"] == "invalid"][["modification", "invalid_reason"]]
      .head(4).to_string(index=False))

n_same = (sug["category"] == "unchanged").sum()
print(f"\n--- discarded: {n_same} suggestions described a change and made none ---")

print("\n--- discarded: reasoning about structures that are not there ---")
ung = sug[sug["describes_other_molecule"] == True]
print(ung["ungrounded_claims"].value_counts().head(6).to_string())
for _, r in ung.head(2).iterrows():
    print(f"\n  claimed:  {r['modification']}")
    print(f"  not in the molecule:  {r['ungrounded_claims']}")

---

## How this works, in full

### The problem it is built around

Ask a language model to improve a drug candidate and you get a confident answer.
The hard part is knowing whether the answer is any good.

The field's usual solution is to score proposals with a *prediction model* — an
ADMET or affinity predictor. That is circular: the generator learns to satisfy the
predictor, the benchmark uses the same predictor, and nobody can tell real chemistry
apart from gaming the scorer.

This tool avoids the circle by going backwards in time. Medicinal chemists have
already run thousands of optimization campaigns and published them. In a finished
campaign **the answer is known and it was measured, not predicted**. So: hide the
ending, hand over the beginning, and compare.

### Where the questions come from

Activity data is pulled from ChEMBL and filtered down to series that are genuine
optimization campaigns:

* one protein target, direct and unambiguous (ChEMBL confidence score 9)
* one assay, so the potencies are comparable to each other
* ≥20 compounds with mean pairwise Tanimoto > 0.40 — a congeneric series, not an
  unrelated pile of hits
* ≥2 log units of activity range, so something was actually optimized

14 targets were surveyed; 716 series passed. 32 of them, across BRD4, HDAC1,
thrombin and EGFR, make up the panel in this notebook.

### A worked example

`CHEMBL6106211_IC50` — an HDAC1 series from a 2025 paper, 31 compounds spanning
pIC50 **5.31 to 8.47**. Every compound shares one architecture: a cap group, an aryl
ether, a pyridine–pyrazole core, an alkyl linker, and a hydroxamic acid as the
zinc-binding group.

The starting compound is taken from the weak end — pIC50 5.46, carrying a bulky
*tert*-butyl-isoxazole cap and an eleven-carbon linker. The 26 compounds that beat
it by ≥1 log unit become the answer key.

What the chemists actually did, visible in that key: **drop the bulky isoxazole cap**
for a simple urea, and **tune the linker down to about seven carbons**. That reaches
pIC50 8.47 — a thousand-fold gain.

### What a model is given

Only the protein name, the assay endpoint, the starting SMILES and its measured
potency — then *"propose 5 modifications and explain your reasoning"*. No answer
compounds, no count of them, no hint of how much gain is available. Every run audits
all 32 real prompts for leakage.

The reasoning is a **separate required field** from the molecule, because a
suggestion can be a change chemists really made while the stated reason is invented.
Telling those two apart is the entire point.

### How the thresholds are set

The two numbers that could have been arbitrary are read off the series itself:

* the **similarity floor** is the *least similar* compound the real chemists made
  from that lead. A proposal only counts as abandoning the scaffold if it strays
  further than the real campaign ever did.
* **"a modification that raises potency"** is not a curated belief about chemistry —
  it is the set of functional-group changes seen between the lead and the real
  improved compounds in that same series.

### What it refuses to do

A valid, on-scaffold molecule that simply never appears in the series is labelled
`novel_plausible` and **left unscored** — the blue verdict. Nobody made that compound,
so there is no measurement, and inventing one with a predictor would rebuild the
circular problem above. Those are the suggestions a human has to judge.

### Why the reference arms exist

A single number is uninterpretable. *"The model recovered 12% of the real
improvements"* means nothing until you know what rote advice and random chemistry
score on the same 32 questions. Five reference arms answer the identical questions,
including an **oracle** handed the real answers (it scores exactly 1.000, which is
how we know the scoring can reach its ceiling) and a **generic** arm that proposes
the same five substituents to every molecule forever.

That calibration changed how the results must be read: the generic arm scores
**0.607** on change-type accuracy, because halogen scanning genuinely does raise
potency. A model scoring around 0.6 there has demonstrated nothing.

### What was found so far

Gemma 3 12B, a local open-weight model, over all 32 series (160 suggestions):
recovered **0** of the real improvements; change-type accuracy **indistinguishable**
from every reference arm (p = 0.17–0.78); 32% of its proposals were not valid
molecules and another 9% were the starting compound returned unchanged. A third of
its rationales discussed structures absent from the molecule.

On this very HDAC1 series it proposed edits to a "piperidine" and a "quinoline" —
neither present anywhere in the compound — while actually deleting an oxygen from
the isoxazole ring. It never mentioned the hydroxamic acid, the one group an HDAC
chemist looks at first.

**Scope:** that is evidence about 12B-scale open models. A frontier-model arm is
wired up in the code but has not been run.

Full detail, including three metric definitions that had to be corrected after
measurement: [the repository](https://github.com/SinanGokmen/leadopt-bench).